# Local Source Projection — Full Pipeline & Paper Figures

Single, self-contained notebook: **Part 1** generates all data (observations, peak detection,
trajectory loading, segmentation, kernel fitting) for every available peak. **Part 2** builds
Figures 1-5 from that data. No external intermediate files needed beyond the raw inputs listed below.

**Runtime note:** Part 1 processes every peak with locally available trajectory data
(~130s/peak in testing) -- expect ~15-30 minutes total depending on peak count and machine.
Progress is printed per peak.

**Required inputs** (paths configured in `config.json` / Part 1 below):
- `df_dens.parquet`, `site_params.pkl` -- aggregated trajectory/site metadata
- `<traj_dir>/<time>-<site_id>-total-column/about.json` + `traj/particle_stilt.*.parquet` per peak
- the raw EM27 retrieval-bundle parquet (observations)
- the averaging-kernel JSON

# Part 1 — Generate data

## 1.1 Setup: imports, config, base data

In [1]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy.signal import convolve
from pvlib.solarposition import get_solarposition

from lsp_functions import (
    haversine_distance, calculate_initial_bearing, circular_mean,
    pressure_std_atm, molec_column, load_ak,
    calculate_wind_vectors, interpolate_particle_trajectories, read_traj_parq,
    create_radius, bearing_segmentation, ring_area, make_histogram,
    read_obs_proffast_parquet, find_group_peaks, find_nearest,
)
%matplotlib qt
warnings.filterwarnings('ignore')

with open('config.json') as f:
    CONFIG = json.load(f)

CONFIG['data_dir'] = './demo_data_allpeaks'
CONFIG['traj_dir'] = f"{CONFIG['data_dir']}/traj"
CONFIG['df_dens_path'] = f"{CONFIG['data_dir']}/df_dens.parquet"
CONFIG['site_params_path'] = f"{CONFIG['data_dir']}/site_params.pkl"
CONFIG['averaging_kernel_path'] = f"{CONFIG['data_dir']}/ma_avk_CH4_2020.json"
CONFIG['obs_bundle_path'] = f"{CONFIG['data_dir']}/em27-retrieval-bundle-ma-proffast-2_4-GGG2020-20161102-20161104.parquet"
CONFIG['obs_location_id'] = 'SF_LAL'  # same physical site as SF_LAB, different location_id label
CONFIG['utc_offset_hours'] = -7       # PDT; used only for the local calendar-day boundary

site_dw = CONFIG['site_dw']
site_id = CONFIG['site_id']

df_dens = pd.read_parquet(CONFIG['df_dens_path'])
site_params = pd.read_pickle(CONFIG['site_params_path'])

print(f"df_dens: {df_dens.shape}")

df_dens: (3527, 13)


## 1.2 Observations and peak detection (from the raw retrieval bundle)

In [2]:
observations = read_obs_proffast_parquet(
    CONFIG['obs_bundle_path'], date=CONFIG['date'], species='CH4',
    quantile=CONFIG['quantile'], roll_time=CONFIG['roll_time'],
    location_id=CONFIG['obs_location_id'], utc_offset_hours=CONFIG['utc_offset_hours'],
)
observations = observations.reset_index().set_index('minutes')


peak_data = observations.groupby('nan_chunk_index').apply(
    lambda x: find_group_peaks(x, key='Enh_ppm', prominence=CONFIG['prominence']), include_groups=False
)
minutes_grid = np.unique(df_dens.minutes)
peak_data['peak_time_grid'] = [find_nearest(minutes_grid, x) for x in peak_data.peak_time]
peak_data = peak_data.reset_index().set_index('minutes')

print(f"observations: {observations.shape}, peaks found: {len(peak_data)}")

observations: (2210, 20), peaks found: 38


## 1.3 Averaging-kernel weighting (paper Appendix D)

In [3]:
la_0 = site_params[site_dw]['lati_0']
lo_0 = site_params[site_dw]['long_0']

la_0 = 37.875778811697415
lo_0 = -122.25742163095923

m_air = 28.97 / 1000

alts = np.unique(df_dens.alt)
diffs = np.diff(np.concatenate(([-alts[0]], alts)))
layers = [0.5 * (diffs[i + 1] + diffs[i]) for i in range(len(alts) - 1)]
layers.append(layers[-1])
layer_fun = {alts[i]: layers[i] for i in range(len(alts))}
df_dens['layer_thickness'] = df_dens['alt'].apply(lambda x: layer_fun[x])

df_dens['utc'] = df_dens['recep'].apply(str).apply(lambda x: pd.to_datetime(x, format='%Y%m%d%H%M', utc=True))
df_dens['sza'] = 90 - get_solarposition(
    df_dens['utc'].to_numpy(), la_0, lo_0, altitude=None, pressure=None,
    method='nrel_numpy', temperature=12.0
)['apparent_elevation'].to_numpy()

pressures = pressure_std_atm(df_dens['alt'] + 124)
ak = load_ak(CONFIG['averaging_kernel_path'])
df_dens['ak'] = ak((df_dens['sza'], pressures))

df_dens['layer_molec_m2'] = df_dens['density'] / m_air * df_dens['layer_thickness'] / 1000
df_dens['dens_layer_weight'] = df_dens['layer_molec_m2'] / molec_column(124)
df_dens['full_weight_per_trajectory'] = df_dens['dens_layer_weight'] / df_dens['numpar'] * df_dens['ak']

print('Averaging-kernel weighting complete.')

Averaging-kernel weighting complete.


## 1.4 Which peak to inspect in detail

Figures 2 and 4 need one peak's full segment-level detail (including the fitted time series
for the inset plots). Figure 5 and the residual-vs-distance validation use **all** peaks.
Set `SELECTED_PEAKTIME_GRID` below (or leave as the first available peak).

In [4]:
def available_peaktime_grids(config, peak_data):
    """Peaks for which a trajectory folder was actually exported to disk."""
    out = []
    for ptg in np.unique(peak_data.peak_time_grid):
        ctime = pd.to_datetime(config['date']) + pd.to_timedelta('%imin' % int(ptg))
        p = f"{config['traj_dir']}/{ctime.strftime('%Y%m%d-%H%M')}-{config['site_id']}-total-column/about.json"
        if os.path.exists(p):
            out.append((ptg, ctime))
    return out

PEAK_OPTIONS = available_peaktime_grids(CONFIG, peak_data)
print(f"{len(PEAK_OPTIONS)} peak(s) with local trajectory data:")
for ptg, ctime in PEAK_OPTIONS:
    row = peak_data[peak_data.peak_time_grid == ptg].iloc[0]
    print(f"  peaktime_grid={ptg:>7.1f}  ({ctime.strftime('%H:%M')} UTC)  peak height: {row.peak_height*1000:.1f} ppb")

# --- SELECT PEAK HERE ---
SELECTED_PEAKTIME_GRID = PEAK_OPTIONS[0][0]  # e.g. set explicitly: SELECTED_PEAKTIME_GRID = 990.0
# ------------------------
print(f"\nSelected for detailed Figures 2/4: peaktime_grid={SELECTED_PEAKTIME_GRID}")

13 peak(s) with local trajectory data:
  peaktime_grid=  990.0  (16:30 UTC)  peak height: 10.4 ppb
  peaktime_grid= 1005.0  (16:45 UTC)  peak height: 7.3 ppb
  peaktime_grid= 1020.0  (17:00 UTC)  peak height: 9.5 ppb
  peaktime_grid= 1035.0  (17:15 UTC)  peak height: 5.1 ppb
  peaktime_grid= 1050.0  (17:30 UTC)  peak height: 9.5 ppb
  peaktime_grid= 1065.0  (17:45 UTC)  peak height: 7.8 ppb
  peaktime_grid= 1080.0  (18:00 UTC)  peak height: 9.1 ppb
  peaktime_grid= 1095.0  (18:15 UTC)  peak height: 4.9 ppb
  peaktime_grid= 1125.0  (18:45 UTC)  peak height: 5.3 ppb
  peaktime_grid= 1140.0  (19:00 UTC)  peak height: 5.0 ppb
  peaktime_grid= 1200.0  (20:00 UTC)  peak height: 5.9 ppb
  peaktime_grid= 1215.0  (20:15 UTC)  peak height: 5.8 ppb
  peaktime_grid= 1245.0  (20:45 UTC)  peak height: 7.8 ppb

Selected for detailed Figures 2/4: peaktime_grid=990.0


## 1.5 Process every available peak

For each peak: load trajectories, segment the upwind domain, and fit the transport kernel
per segment (Step 1 & 2, paper Eq. 5-8). The **selected** peak's full result (including
per-segment time series, for the Fig. 2/4 insets) is kept in `trajectories` / `fitted_to_obs`.
A slimmed-down summary (dropping the bulky per-segment time series) for **every** peak is
concatenated into `all_fits`, used by Figure 5 and the multi-peak residual validation.

This is the slow step (see runtime note at the top).

In [ ]:
radius_steps_km, delta_alpha_deg = create_radius(
    CONFIG['drf'], dr=CONFIG['dr'], rad_max=CONFIG['rad_max_km'], max_steps=CONFIG['max_radial_steps'], d_alpha=True
)
source_altitude_segmentation = np.linspace(0, CONFIG['max_source_altitude_m'], CONFIG['source_altitude_bins'])

ARRAY_COLS = ['observations_ppm', 'kernel_fitted', 'duration_minutes', 'duration_residual', 'time_minutes']

all_fits_list = []
fitted_by_peak = {}   # all fit results aggregated not only selected one.
trajectories = None   # full segmented trajectories, kept only for the selected peak
fitted_to_obs = None  # full per-segment fit results (with array columns), selected peak only

t_start = time.time()
for n, (peaktime_grid, ctime) in enumerate(PEAK_OPTIONS):
    t0 = time.time()
    traj_path = f"{CONFIG['traj_dir']}/{ctime.strftime('%Y%m%d-%H%M')}-{site_id}-total-column/"

    traj_column, meta = read_traj_parq(
        traj_path, select_release_heights=range(CONFIG['release_heights']),
        cutoff_time=CONFIG['trajectory_cutoff_minutes'], hi_res_time=CONFIG['hi_res_time_resolution'],
        averaging_kernel_path=CONFIG['averaging_kernel_path'],
    )

    cur_traj = traj_column.copy().reset_index(drop=True)
    cur_traj = cur_traj[cur_traj.recep_dist_km < 10].reset_index(drop=True)
    cur_traj['recep_dist_segmentation'] = pd.cut(cur_traj['recep_dist_km'], radius_steps_km, right=False)
    cur_traj['source_agl_segmentaion'] = pd.cut(cur_traj.zagl, source_altitude_segmentation, right=False)
    intervals = pd.IntervalIndex.from_breaks(source_altitude_segmentation, closed='left')
    dz_dict = {interval: interval.right - interval.left for interval in intervals}
    cur_traj['dz_source'] = cur_traj['source_agl_segmentaion'].map(dz_dict)
    cur_traj['dalpha'] = np.interp(cur_traj['recep_dist_km'], radius_steps_km,
                                    np.concatenate([delta_alpha_deg[1:], [delta_alpha_deg[-1]]]))
    a = cur_traj.groupby(['source_agl_segmentaion', 'recep_dist_segmentation'], observed=True).apply(
        bearing_segmentation, include_groups=True
    )
    a_flat = a.reset_index(level=[0, 1], drop=True)
    cur_traj = cur_traj.join(a_flat[['recep_bearing_segmentation', 'deg_step']], how='inner')
    cur_traj['segment_area_m2'] = ring_area(cur_traj['recep_dist_segmentation'], cur_traj['recep_bearing_segmentation'])

    peak_row = peak_data[peak_data.peak_time_grid == peaktime_grid].iloc[0]
    peak_time = peak_row.peak_time
    pw = CONFIG['peak_window_minutes']
    peak_section = observations[(observations.index <= peak_time + pw / 2) & (observations.index > peak_time - pw / 2)]

    cur_fitted = cur_traj.groupby(
        ['source_agl_segmentaion', 'recep_dist_segmentation', 'recep_bearing_segmentation'], observed=True
    ).apply(
        lambda x: make_histogram(
            x, peak_section, peak_time, std_dev=2, weight=False, plot=False,
            max_plot_radius=0.3, debug=False, emission_duration=True,
            hi_res_time_resolution=CONFIG['hi_res_time_resolution'],
        ),
        include_groups=False,
    ).reset_index()

    cur_fitted['recep_dist_km'] = cur_fitted['recep_dist_segmentation'].apply(lambda x: 0.5 * (x.left + x.right)).astype(float)
    cur_fitted['recep_bearing_deg_mid'] = cur_fitted['recep_bearing_segmentation'].apply(
        lambda x: 0.5 * (x.left + x.right) if pd.notna(x) else np.nan).astype(float)
    cur_fitted['duration_residual_std_ppm'] = cur_fitted['duration_residual'].apply(lambda x: np.nanstd(x))
    cur_fitted['peak_time'] = peak_time
    cur_fitted['peaktime_grid'] = peaktime_grid
    
    fitted_by_peak[peaktime_grid] = cur_fitted.copy()

    if peaktime_grid == SELECTED_PEAKTIME_GRID:
        trajectories = cur_traj
        fitted_to_obs = cur_fitted.copy()

    #all_fits_list.append(cur_fitted.drop(columns=ARRAY_COLS))
    all_fits_list.append(cur_fitted)

    dt = time.time() - t0
    print(f"[{n+1}/{len(PEAK_OPTIONS)}] {ctime.strftime('%H:%M')} UTC done in {dt:.0f}s "
          f"(total elapsed {(time.time()-t_start)/60:.1f} min), {len(cur_fitted)} segments")

all_fits = pd.concat(all_fits_list, ignore_index=True)
print(f"\nDONE. Total time: {(time.time()-t_start)/60:.1f} min. all_fits shape: {all_fits.shape}")

In [5]:
import pickle
from pathlib import Path
import datetime

def save_results(
    filepath,
    all_fits,
    fitted_by_peak,
    trajectories,
    fitted_to_obs,
    radius_steps_km,
    delta_alpha_deg,
    source_altitude_segmentation,
    selected_peaktime_grid,
    #array_cols=ARRAY_COLS,
    config=None,
):
    """Speichert alle Ergebnisse der PEAK_OPTIONS-Iteration in einer Pickle-Datei."""
    payload = {
        'all_fits': all_fits,
        'fitted_by_peak': fitted_by_peak,
        'trajectories': trajectories,
        'fitted_to_obs': fitted_to_obs,
        'radius_steps_km': radius_steps_km,
        'delta_alpha_deg': delta_alpha_deg,
        'source_altitude_segmentation': source_altitude_segmentation,
        'selected_peaktime_grid': selected_peaktime_grid,
        #'array_cols': array_cols,
        'config': config,  # optional: CONFIG-Dict mitspeichern für Nachvollziehbarkeit
    }

    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'wb') as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Ergebnisse gespeichert: {filepath} ({filepath.stat().st_size / 1e6:.1f} MB)")


def load_results(filepath):
    """Lädt die zuvor gespeicherten Ergebnisse und gibt sie als Dict zurück."""
    filepath = Path(filepath)
    with open(filepath, 'rb') as f:
        payload = pickle.load(f)

    print(f"Ergebnisse geladen: {filepath}")
    print(f"  all_fits shape: {payload['all_fits'].shape}")
    print(f"  fitted_by_peak: {len(payload['fitted_by_peak'])} peaks")
    print(f"  selected_peaktime_grid: {payload['selected_peaktime_grid']}")

    return payload

# Dual-Zeit-Formatierung (nur auf unterer Achse nötig, da sharex=True)
def dual_time_formatter(x, pos):
    dt_utc = mdates.num2date(x).replace(tzinfo=None)
    dt_local = dt_utc + pd.Timedelta(hours=CONFIG['utc_offset_hours'])
    return f"{dt_utc.strftime('%H:%M')}\n({dt_local.strftime('%H:%M')} PDT)"




def apply_wind_speed_correction(
    df,
    met_path,
    reference_candidate='LBNL1',
    candidate_col='candidate',
    peak_time_col='peak_time',
    modeled_wind_col='wind_speed',
    emission_cols=None,
    suffix='_wcorr',
    drop_reference=True,
    inplace=False,
):
    """
    Ermittelt den Windgeschwindigkeits-Korrekturfaktor (gemessen/modelliert)
    lokal am Referenzstandort (z.B. LBNL1, wo die Met-Station steht) pro
    peak_time, und wendet ihn auf alle Kandidaten an (inkl. Referenz selbst,
    falls drop_reference=False).

    Parameters
    ----------
    df : DataFrame mit `candidate_col`, `peak_time_col`, `modeled_wind_col`
    met_path : Pfad zur Berkeley-Met-CSV
    reference_candidate : Name des Referenz-'Kandidaten' (z.B. 'LBNL1'),
                          der die gemessene Windgeschwindigkeit liefert
    drop_reference : bei True wird der Referenzkandidat aus dem Ergebnis
                     entfernt (da er kein echter Emissions-Kandidat ist)

    Returns
    -------
    df mit zusätzlichen Spalten <col><suffix>, 'wind_speed_measured' und
    'wind_speed_corr_factor' (letztere übertragen von LBNL1 auf alle
    anderen Kandidaten, gematcht über peak_time_col).
    """
    met = read_berkeley_met(path=met_path)
    met_spd = lambda x: np.interp(x, met.minutes, met.wind_speed_set_1)

    out = df if inplace else df.copy()

    ref = out[out[candidate_col] == reference_candidate].copy()
    if ref.empty:
        raise ValueError(f"Referenzkandidat '{reference_candidate}' nicht in '{candidate_col}' gefunden.")

    ref['wind_speed_measured'] = met_spd(ref[peak_time_col])
    ref['wind_speed_corr_factor'] = ref['wind_speed_measured'] / ref[modeled_wind_col]

    # Korrekturfaktor pro peak_time (LBNL1-Referenz), auf alle Kandidaten übertragen
    corr_map = ref.set_index(peak_time_col)['wind_speed_corr_factor']
    out['wind_speed_corr_factor'] = out[peak_time_col].map(corr_map)

    if emission_cols is None:
        emission_cols = [
            c for c in out.columns
            if c in ['emission_mol','emission_mean','emission_median','emission_min','emission_max'] or c.endswith(('_g_s', '_kg_peak', '_tCH4_yr'))
        ]

    for col in emission_cols:
        out[f'{col}{suffix}'] = out[col] * out['wind_speed_corr_factor']

    if drop_reference:
        out = out[out[candidate_col] != reference_candidate].reset_index(drop=True)

    return out


def read_berkeley_met(path ='D:\\inverse_modeling_python\\LocalSourceProjection\\LBNLmet\\LBNL1_20161103.csv',plot=False):
    met = pd.read_csv(path,header=[6,7])
    #path = 'C:\\inverse_modeling_python\\LocalSourceProjection\\LBNL1.csv'
    met.columns = met.columns.droplevel(1)
    met['pressure_hpa']=met.pressure_set_1/0.029529983071445
    alt = 270.6 #ASL
    
    
    met['date'] = [datetime.datetime.strptime(x,'%m/%d/%Y %H:%M UTC') for x in np.array(met['Date_Time'])]
    ref_date = met['date'][0]
    met['hour'] = [(x-ref_date).total_seconds()/60/60 for x in met.date]
    met['minutes'] = met['hour']*60
    
    if plot:
        delta_p = np.exp(-(124-270)/8400)


        z  = np.array(zagls)+np.array(zsfcs)-271
        zagl_lbnl = 26
        z_0 = 1.0
        v_corr = np.array(spd_collector)/np.log(np.array(zagls)/z_0)*np.log(zagl_lbnl/z_0)
        
        fig, axs = plt.subplots(nrows=2,sharex=True,gridspec_kw={'height_ratios': [3, 1]})
        axs[0].plot(met.hour,met.loc[:,(             'wind_direction_set_1',            'Degrees')],label='observed')
        axs[0].plot(cdf.index/60,dir_collector,label='HRRR-STILT')
        axs[0].set_ylim((0,160))
        axs[1].plot(cdf.index/60,dist_collector,'r',label='horizontal')
        axs[1].plot(cdf.index/60,z,'b',label='vertical')
        axs[1].legend()
        axs[0].legend()
        axs[0].set_ylabel('wind direction / deg')
        axs[1].set_xlabel('hour / utc')
        axs[1].set_ylabel('dist / m')
        axs[1].set_ylim((0,300))
        plt.xlim((16,24))

        fig.suptitle('LBNL1 Berkeley Lab\nLAT: 37.8771 LON: -122.2486')
        

        fig, axs = plt.subplots(nrows=2,sharex=True,gridspec_kw={'height_ratios': [3, 1]})
        axs[0].plot(met.hour,met.loc[:,(             'wind_speed_set_1',            'm/s')],label='observed')
        axs[0].plot(cdf.index/60,spd_collector,label='HRRR-STILT')
        axs[0].plot(cdf.index/60,v_corr,label='HRRR-STILT - altitude adjusted')
        
        axs[0].legend()
        axs[0].set_ylabel('wind speed / m/s')


        axs[1].plot(cdf.index/60,dist_collector,'r',label='horizontal')
        axs[1].plot(cdf.index/60,z,'b',label='vertical')
        axs[1].legend()
        axs[1].set_xlabel('hour / utc')
        axs[1].set_ylabel('dist / m')
        axs[1].set_ylim((0,300))
        plt.xlim((16,24))

        fig.suptitle('LBNL1 Berkeley Lab\nLAT: 37.8771 LON: -122.2486')
    
    
    
    return(met)

In [ ]:
save_results(
    filepath=f"{CONFIG['output_dir']}/fit_results_{site_id}.pkl",
    all_fits=all_fits,
    fitted_by_peak=fitted_by_peak,
    trajectories=trajectories,
    fitted_to_obs=fitted_to_obs,
    radius_steps_km=radius_steps_km,
    delta_alpha_deg=delta_alpha_deg,
    source_altitude_segmentation=source_altitude_segmentation,
    selected_peaktime_grid=SELECTED_PEAKTIME_GRID,
    config=CONFIG,
)

In [6]:
results = load_results(f"{CONFIG['output_dir']}/fit_results_{site_id}.pkl")

all_fits = results['all_fits']
fitted_by_peak = results['fitted_by_peak']
trajectories = results['trajectories']
fitted_to_obs = results['fitted_to_obs']
radius_steps_km = results['radius_steps_km']
delta_alpha_deg = results['delta_alpha_deg']
source_altitude_segmentation = results['source_altitude_segmentation']
SELECTED_PEAKTIME_GRID = results['selected_peaktime_grid']

del results

Ergebnisse geladen: demo_output\fit_results_SF_LAB.pkl
  all_fits shape: (14301, 103)
  fitted_by_peak: 13 peaks
  selected_peaktime_grid: 990.0


# Part 2 — Paper figures

## Figure 1: observed enhancement and detected peaks

In [7]:

UTC_OFFSET_HOURS = CONFIG['utc_offset_hours']
base = pd.to_datetime(CONFIG['date'])
obs_time = base + pd.to_timedelta(observations.index, unit='m')
peak_time_dt = base + pd.to_timedelta(peak_data.peak_time, unit='m')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(obs_time, observations.Enh_ppm * 1000, lw=0.7, label='observations', color='tab:blue')
ax.plot(peak_time_dt, peak_data.peak_height * 1000, 'ro', label='peak', ms=4)

line_starts = np.arange(peak_data.peak_time.min(), peak_data.peak_time.max(), 12)
for x in line_starts:
    ax.axvline(base + pd.to_timedelta(x, unit='m'), lw=0.2, color='k')
ax.axvline(base + pd.to_timedelta(line_starts[-1], unit='m'), lw=0.2, color='k', label='12 min interval')

def dual_time_formatter(x, pos):
    dt_utc = mdates.num2date(x).replace(tzinfo=None)
    dt_local = dt_utc + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    return f"{dt_utc.strftime('%H:%M')}\n({dt_local.strftime('%H:%M')} PDT)"

ax.xaxis.set_major_formatter(FuncFormatter(dual_time_formatter))
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
ax.set_xlim(base + pd.Timedelta('16:15:00'), base + pd.Timedelta('19:30:00'))
ax.set_xlabel('UTC (local time)')
ax.set_ylabel('Enhancement XCH4 / ppb')
ax.legend()
plt.tight_layout()
plt.savefig('Figure1.png', dpi=250)
print('saved Figure1.png')

saved Figure1.png


In [8]:

observations_prf_2_2 = read_obs_proffast_parquet(
    CONFIG['obs_bundle_path'].replace('-2_4-','-2_2-'), date=CONFIG['date'], species='CH4',
    quantile=CONFIG['quantile'], roll_time=CONFIG['roll_time'],
    location_id=CONFIG['obs_location_id'], utc_offset_hours=CONFIG['utc_offset_hours'],
)
observations_prf_2_2 = observations_prf_2_2.reset_index().set_index('minutes')

peak_data_2_2 = observations_prf_2_2.groupby('nan_chunk_index').apply(
    lambda x: find_group_peaks(x, key='Enh_ppm', prominence=CONFIG['prominence']), include_groups=False
)
minutes_grid = np.unique(df_dens.minutes)
peak_data_2_2['peak_time_grid'] = [find_nearest(minutes_grid, x) for x in peak_data_2_2.peak_time]
peak_data_2_2 = peak_data_2_2.reset_index().set_index('minutes')

print(f"observations: {observations_prf_2_2.shape}, peaks found: {len(peak_data_2_2)}")

UTC_OFFSET_HOURS = CONFIG['utc_offset_hours']
base = pd.to_datetime(CONFIG['date'])

# --- Gemeinsamen Zeitbereich (minutes) bestimmen und beide Versionen zuschneiden ---
common_min = max(observations_prf_2_2.index.min(), observations.index.min())
common_max = min(observations_prf_2_2.index.max(), observations.index.max())

obs_2_2 = observations_prf_2_2.loc[common_min:common_max].sort_index()
obs_2_4 = observations.loc[common_min:common_max].sort_index()

# Sicherstellen, dass beide exakt dieselben minutes-Werte haben (Sampling ist laut Annahme identisch)
common_idx = obs_2_2.index.intersection(obs_2_4.index)
obs_2_2 = obs_2_2.loc[common_idx]
obs_2_4 = obs_2_4.loc[common_idx]

# Differenz: 2.4.1 - 2.2
diff_enh = (obs_2_4['Enh_ppm'] - obs_2_2['Enh_ppm']) * 1000  # ppb

# --- Zeitachsen für Plot ---
time_2_2 = base + pd.to_timedelta(observations_prf_2_2.index, unit='m')
time_2_4 = base + pd.to_timedelta(observations.index, unit='m')
time_diff = base + pd.to_timedelta(common_idx, unit='m')

peak_time_2_2 = base + pd.to_timedelta(peak_data_2_2.peak_time, unit='m')
peak_time_2_4 = base + pd.to_timedelta(peak_data.peak_time, unit='m')

# --- Plot: oben Vergleich (2/3), unten Differenz (1/3), gemeinsame x-Achse ---
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(9, 6), sharex=True,
    gridspec_kw={'height_ratios': [2, 1]}
)

# Oben: beide Enhancement-Versionen (+ Peaks, ggf. später entfernbar)
ax1.plot(time_2_2, observations_prf_2_2.Enh_ppm * 1000, lw=0.7, label='prf 2.2', color='tab:orange')
ax1.plot(time_2_4, observations.Enh_ppm * 1000, lw=0.7, label='prf 2.4.1', color='tab:blue')
ax1.plot(peak_time_2_2, peak_data_2_2.peak_height * 1000, 'o', color='tab:orange', ms=4, label='peak 2.2')
ax1.plot(peak_time_2_4, peak_data.peak_height * 1000, 'o', color='tab:blue', ms=4, label='peak 2.4.1')
ax1.set_ylabel('Enhancement XCH4 / ppb')
ax1.legend()

(peak_data.peak_height-peak_data_2_2.peak_height)/peak_data.peak_height

# Unten: Differenz
ax2.plot(time_diff, diff_enh, lw=0.7, color='tab:green')
ax2.axhline(0, lw=0.5, color='k')
ax2.set_ylabel('Δ Enh. (2.4.1 − 2.2) / ppb')



ax2.xaxis.set_major_formatter(FuncFormatter(dual_time_formatter))
ax2.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
ax2.set_xlim(base + pd.Timedelta('16:15:00'), base + pd.Timedelta('19:30:00'))
ax2.set_xlabel('UTC (local time)')

plt.tight_layout()
plt.savefig('Figure1_difference.png', dpi=250)
print('saved Figure1_difference.png')

change=(peak_data.peak_height.to_numpy()[:12]-peak_data_2_2.peak_height.to_numpy()[:12])/peak_data.peak_height.to_numpy()[:12]
print('max change: %1.3f'%np.max(np.abs(change)*100))
print('median change: %1.3f'%np.median(change*100))

observations: (2202, 20), peaks found: 35
saved Figure1_difference.png
max change: 6.485
median change: -2.903


## Figure 2: particle segmentation illustration

Built from the **selected peak** (`trajectories` / `fitted_to_obs` from Part 1). `ALT_LAYER_INDEX`
picks which single altitude layer is shown in the map; colors there distinguish distance-azimuth
segments *within* that layer only (see paper caption), not altitude. The three example segments
(closest/middle/farthest well-sampled) are chosen automatically.

In [9]:
ALT_LAYER_INDEX = 8
MAX_RAD_KM = 3

alt_bins = sorted(trajectories['source_agl_segmentaion'].cat.categories)
alt = alt_bins[ALT_LAYER_INDEX]
cdf = trajectories[(trajectories.source_agl_segmentaion == alt) & (trajectories.recep_dist_km < MAX_RAD_KM)].copy()
cfit = fitted_to_obs[(fitted_to_obs.source_agl_segmentaion == alt) & (fitted_to_obs.recep_dist_km < MAX_RAD_KM)].copy()
cfit = cfit[cfit['hist_points'] > 0].set_index(['recep_dist_segmentation', 'recep_bearing_segmentation'])

print(f"Altitude layer: {alt} ({len(cdf)} particle-timesteps, {len(cfit)} fitted segments)")

Altitude layer: [80.0, 90.0) (58213 particle-timesteps, 37 fitted segments)


In [10]:
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

fig = plt.figure(figsize=(16 / 1.2, 9 / 1.2))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 3])
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax3 = fig.add_subplot(gs[2, 0], sharex=ax1)
for ax in [ax1, ax2]:
    plt.setp(ax.get_xticklabels(), visible=False)
ax_big = fig.add_subplot(gs[:, 1])
ax_big.set_aspect(1)

colors = {}
for name, group in cdf.groupby(['recep_dist_segmentation', 'recep_bearing_segmentation'], observed=True):
    sc = ax_big.scatter(group.recep_dist_east_km, group.recep_dist_north_km, s=3)
    if name in cfit.index:
        colors[name] = sc.get_facecolor()

ax_big.set_xlabel('receptor distance east / km')
ax_big.set_ylabel('receptor distance north / km')
ax_big.yaxis.set_label_position('right')
ax_big.yaxis.tick_right()

to_ppb = 1000
cdf_idx = cdf.set_index(['recep_dist_segmentation', 'recep_bearing_segmentation'])

valid_names = [n for n in colors.keys() if cfit.loc[n, 'hist_points'] > 100]
valid_sorted = sorted(valid_names, key=lambda n: cdf_idx.loc[n, 'recep_dist_km'].mean()
                       if hasattr(cdf_idx.loc[n, 'recep_dist_km'], 'mean') else cdf_idx.loc[n, 'recep_dist_km'])
picks = [valid_sorted[3], valid_sorted[int(np.floor(len(valid_sorted) / 1.5))], valid_sorted[-2]]

axes = [ax1, ax2, ax3]
time_center = None
points_ax, points_big = [], []

for ax, cname in zip(axes, picks):
    cf = cfit.loc[cname]
    upwind_dist = cdf_idx.loc[cname].recep_dist_km.mean()
    c = colors[cname]
    if time_center is None:
        time_center = cf.time_minutes[np.argmax(cf.observations_ppm)]
    if ax == ax1:
        ax.plot((cf.time_minutes - time_center) * 60, cf.kernel_fitted * to_ppb, lw=6, color=c, label='kernel %1.1fm' % (upwind_dist*1000))
    else:
        ax.plot((cf.time_minutes - time_center) * 60, cf.kernel_fitted * to_ppb, lw=6, color=c, label='kernel %1.1fkm' % upwind_dist)
    ax.plot((cf.time_minutes - time_center) * 60,cf.observations_ppm * to_ppb, linestyle = ':', color='k' ,label='observed')
    points_ax.append((((cf.time_minutes - time_center) * 60)[np.argmax(cf.kernel_fitted)], np.max(cf.kernel_fitted) * to_ppb))
    pt_big = (cdf_idx.loc[cname].recep_dist_east_km.mean(), cdf_idx.loc[cname].recep_dist_north_km.mean())
    points_big.append(pt_big)
    ax_big.scatter(*pt_big, c='k')
    ax.set_ylabel('enhancement / ppb')
    ax.legend(loc=2)
ax3.set_xlabel('time / s')

def trans_fig(ax, x, y):
    bbox = ax.get_position()
    xlm = ax.get_xlim()
    xout = (bbox.x1 - bbox.x0) / (xlm[1] - xlm[0]) * (x - xlm[0]) + bbox.x0
    ylm = ax.get_ylim()
    yout = (bbox.y1 - bbox.y0) / (ylm[1] - ylm[0]) * (y - ylm[0]) + bbox.y0
    return (xout, yout)

plt.tight_layout()
for i, ax in enumerate(axes):
    start = trans_fig(ax, *points_ax[i])
    end = trans_fig(ax_big, *points_big[i])
    arrow = FancyArrowPatch(start, end, transform=fig.transFigure, color='k', arrowstyle='-|>',
                             mutation_scale=15, lw=0.8, connectionstyle='angle,angleA=0,angleB=90,rad=0')
    fig.add_artist(arrow)

plt.savefig('Figure2.png', dpi=250)
print('saved Figure2.png')

saved Figure2.png


In [11]:
fig = plt.figure(figsize=(16 / 1.2, 9 / 1.2))

ax_big = plt.gca()
ax_big.set_aspect(1)

colors = {}
for name, group in cdf.groupby(['recep_dist_segmentation', 'recep_bearing_segmentation'], observed=True):
    sc = ax_big.scatter(group.long, group.lati, s=3)
    if name in cfit.index:
        colors[name] = sc.get_facecolor()

ax_big.set_xlabel('receptor distance east / km')
ax_big.set_ylabel('receptor distance north / km')
ax_big.yaxis.set_label_position('right')
ax_big.yaxis.tick_right()

SOURCE_LOCATIONS = {
    'foothill dining': (37.87544915758012, -122.25616344191205, 'potential candidate, very close'),
    'student housing': (37.87624558858762, -122.25601980030257, 'potential candidate, very close, broken heating system reported'),
    'building 33':     (37.87609216845105, -122.24674281410584, 'smoke stack approximately 40m above ground'),
    'building 30':     (37.87649436515283, -122.24708452824339, 'smoke stack approximately 25m above ground'),
    'ng_infra':        (37.87556109517469, -122.25443161396028, 'natural gas valve, surrounded by trees, injection into atmosphere not likely'),
    'tank':            (37.87846958360055, -122.24143278375334, 'unknown tank, potential large source, injection into atmosphere possible due to height'),
    'LBNL1':           (37.877097985303465, -122.24861278713782, 'location of meteorological site, no source candidate'),
}

for key in SOURCE_LOCATIONS.keys():
    plt.scatter(SOURCE_LOCATIONS[key][1],SOURCE_LOCATIONS[key][0])
    
plt.scatter(lo_0,la_0,s=30,c='r')

## Figure 3: emission / kernel / convolution illustration

Synthetic, independent of the generated data.

In [12]:
t = np.arange(0, 900, 1)
kt = np.arange(-300, 301, 1)

measured_signal = np.exp(-0.5 * ((t - 450) / 20) ** 2)
measured_signal /= measured_signal.max()

def narrow_kernel(t):
    k = np.exp(-0.5 * (t / 10) ** 2)
    return k / k.sum()

def wide_kernel(t):
    k = np.exp(-0.5 * (t / 50) ** 2)
    return k / k.sum()

def very_wide_kernel(t):
    k = np.exp(-0.5 * (t / 100) ** 2)
    return k / k.sum()

kernels = [narrow_kernel(kt), wide_kernel(kt), very_wide_kernel(kt)]

em1 = np.exp(-0.5 * ((t - 450) / 50) ** 2)
em1 *= measured_signal.max() / convolve(em1, kernels[0], mode='same').max()
em2 = np.exp(-0.5 * ((t - 450) / 10) ** 2)
em2 *= measured_signal.max() / convolve(em2, kernels[1], mode='same').max()
em3 = np.zeros_like(t)
em3[450] = 1.0 / 0.004000045722115894

emissions = [em1, em2, em3]
convolutions = [convolve(e, k, mode='same') for e, k in zip(emissions, kernels)]

fig, axes = plt.subplots(3, 3, figsize=(15 / 1.5, 9 / 1.5), sharex='col')
column_titles = ['emission\n[µmol/m²/s]', 'kernel\n[1/(mol/m²/s)]', 'convolution\n[µmol/mol]']
for ax, title in zip(axes[0], column_titles):
    ax.set_title(title)

for i in range(3):
    axes[i][0].plot(t-450, emissions[i], label='emission', color='tab:blue')
    axes[i][0].set_xlim(-300, 300)
    axes[i][1].plot(kt, kernels[i], label='kernel', color='tab:orange')
    axes[i][1].set_xlim(-300, 300)
    axes[i][2].plot(t-450, convolutions[i], label='conv', color='tab:red')  # RC1 fix: was tab:green
    axes[i][2].plot(t-450, convolutions[0], label='observ.', linestyle=':', color='black')
    axes[i][2].set_xlim(-300, 300)
    for j in range(3):
        if i == 2:
            axes[i][j].set_xlabel('t / s')
        axes[i][j].grid(True)

axes[0][2].legend(loc=1)
plt.tight_layout()
plt.subplots_adjust(wspace=0.4)
plt.savefig('Figure3.png', dpi=200)
print('saved Figure3.png')

saved Figure3.png


## Figure 4: residual vs. distance, with inset and emission bar chart

Built from the **selected peak** (`fitted_to_obs` from Part 1).

In [13]:
def plot_figure4(fitted_to_obs, hist_points_min=500, out_path='Figure4.png'):
    fitted_to_obs = fitted_to_obs.copy()
    well_sampled = fitted_to_obs[fitted_to_obs['hist_points'] > hist_points_min].copy()
    if len(well_sampled) == 0:
        raise ValueError(f"No segments with hist_points > {hist_points_min}.")

    cols = ['recep_dist_km', 'residual_std_ppm', 'duration_residual_std_ppm', 'emission_mol', 'hist_points']
    me_df = well_sampled.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanmean)
    st_df = well_sampled.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanstd)
    r_max_km = me_df.loc[me_df['residual_std_ppm'].idxmin(), 'recep_dist_km']

    well_sampled['dist_to_rmax'] = (well_sampled['recep_dist_km'] - r_max_km).abs()
    example_row = well_sampled.sort_values('dist_to_rmax').iloc[0]

    fig, (a0, a1) = plt.subplots(2, 1, sharex=True, figsize=(8, 5.5), gridspec_kw={'height_ratios': [3, 1]})
    a0.errorbar(me_df['recep_dist_km'], me_df['duration_residual_std_ppm'] * 1000, yerr=st_df['duration_residual_std_ppm'] * 1000,
                label='kernel * emissions', fmt='-o', lw=0.5, capsize=3, c='red')
    a0.errorbar(me_df['recep_dist_km'], me_df['residual_std_ppm'] * 1000, yerr=st_df['residual_std_ppm'] * 1000,
                fmt=':d', label='kernel only', capsize=3, lw=0.5, c='orange')
    a0.axvline(r_max_km, color='r', lw=0.8, ls='--', label=f'r_max = {r_max_km:.2f} km')
    a0.set_ylabel('fit residual std / ppb')
    
    peak_time = well_sampled.peak_time.iloc[0]
    ctime = base + pd.Timedelta(minutes=float(peak_time))
    local_ctime = base + pd.Timedelta(minutes=(float(peak_time)+CONFIG['utc_offset_hours']*60))
    a0.set_title(f"{ctime:%H:%M} UTC" + f"  ({local_ctime:%H:%M} PTD)")

    a01 = a0.inset_axes([0.5, 0.0, 0.5, 0.5])
    t_ = example_row['time_minutes']
    a01.plot(t_, example_row['observations_ppm'] + example_row['duration_residual'], c='red', label='kernel * emission', zorder=9)
    a01.plot(t_, example_row['kernel_fitted'], c='orange', label='kernel', zorder=10)
    a01.plot(t_, example_row['observations_ppm'], linestyle=':', color='black', zorder=11, label='observation')
    a01.get_xaxis().set_visible(False)
    a01.get_yaxis().set_visible(False)
    a01.legend(fontsize=6, loc='upper right')
    a0.legend(loc='upper center')

    wdh = np.diff(me_df['recep_dist_km']) * 0.8
    wdh = np.concatenate((wdh, [wdh[-1]]))
    a1.bar(me_df['recep_dist_km'], me_df['emission_mol'], yerr=st_df['emission_mol'], width=wdh)
    a1.set_yscale('log')
    a1.set_ylabel('total emission / mol')
    a1.set_xlabel('upwind distance / km')
    plt.tight_layout()
    plt.savefig(out_path, dpi=250)
    print(f"r_max = {r_max_km:.3f} km, well-sampled segments: {len(well_sampled)} / {len(fitted_to_obs)}")
    return fig, r_max_km

_ = plot_figure4(fitted_to_obs, out_path='.\Figure4.png')

r_max = 0.966 km, well-sampled segments: 262 / 547


## Figure 5: candidate source emission estimates (paper Table 1 / Fig. 5)

Uses `all_fits` from Part 1 -- all peaks processed in this notebook run, no external file needed.
For each candidate location, computes distance and bearing from the receptor, matches it to the
corresponding upwind segment for every available peak, and shows the per-peak, per-candidate
emission estimate. A candidate can be unmatched for a given peak if that peak's wind direction
didn't sample the candidate's bearing.

**Scope note:** extends the paper's original two candidates (Foothill Dining, Student Housing)
to all 7 locations investigated during the source search.

**Important:** the paper's Figure 5 applies a wind-speed correction (observed LBNL1 wind speed
in place of the HRRR model, paper Appendix G), which needs the LBNL1 meteorological file --
not part of this notebook's inputs. The emissions below are **uncorrected**.

In [14]:
SOURCE_LOCATIONS = {
    'foothill dining': (37.87544915758012, -122.25616344191205, 'potential candidate, very close'),
    'student housing': (37.87608233036625, -122.25645436619163, 'potential candidate, very close, broken heating system reported'),
    'NG infrastructure':        (37.87556109517469, -122.25443161396028, 'natural gas valve, surrounded by trees, injection into atmosphere not likely'),
    'building 30':     (37.87649436515283, -122.24708452824339, 'smoke stack approximately 25m above ground'),
    'building 33':     (37.87609216845105, -122.24674281410584, 'smoke stack approximately 40m above ground'),
    'tank':            (37.87846958360055, -122.24143278375334, 'unknown tank, potential large source, injection into atmosphere possible due to height'),
    'LBNL1':           (37.877097985303465, -122.24861278713782, 'location of meteorological site, no source candidate'),
}

def match_candidate(fitted, lat, lon, la_0, lo_0, hist_points_min=500):
    d_km = haversine_distance(la_0, lo_0, lat, lon) / 1000
    bearing = calculate_initial_bearing(la_0, lo_0, lat, lon)
    
    # +/- DEG opens the acceptance-window for the bearing to account for variations in wind-direction
    # 180 corresponds to full circle (all bearings are accepted, just range is relevant)
    # 0 corresponds to a strict macth in bearing.
    bearing_delta = 0 # DEG +/-   

    
    def _to_segments(lo, hi):
        """Zerlegt ein zirkuläres Intervall [lo, hi) (mod 360) in 1-2 lineare Segmente."""
        lo, hi = lo % 360, hi % 360
        if lo < hi:
            return [(lo, hi)]
        elif lo > hi:
            return [(lo, 360.0), (0.0, hi)]
        else:
            # lo == hi: entweder leeres oder volles Intervall -> hier als voll behandeln
            return [(0.0, 360.0)]

    def _linear_overlap(a, b):
        return a[0] < b[1] and b[0] < a[1]

    def circular_overlap(lo1, hi1, lo2, hi2):
        segs1 = _to_segments(lo1, hi1)
        segs2 = _to_segments(lo2, hi2)
        return any(_linear_overlap(s1, s2) for s1 in segs1 for s2 in segs2)

    d_km = haversine_distance(la_0, lo_0, lat, lon) / 1000
    bearing = calculate_initial_bearing(la_0, lo_0, lat, lon)
    

    def in_dist(iv):
        return pd.notna(iv) and iv.left <= d_km < iv.right

    # Sonderfall: Delta deckt den Vollkreis ab (z.B. bearing_delta >= 180)
    full_circle = (2 * bearing_delta) >= 360

    def in_bearing_window(iv):
        if pd.isna(iv):
            return False
        if full_circle:
            return True
        lo_w = bearing - bearing_delta
        hi_w = bearing + bearing_delta
        return circular_overlap(lo_w, hi_w, iv.left, iv.right)

    mask = fitted['recep_dist_segmentation'].apply(in_dist) & fitted['recep_bearing_segmentation'].apply(in_bearing_window)
    matched = fitted[mask]
    matched = matched[matched['hist_points'] > hist_points_min]
    return matched, d_km, bearing

met_agl = np.unique(all_fits['source_agl_segmentaion'])[2]
records = []
for name, (lat, lon, comment) in SOURCE_LOCATIONS.items():
    for peak_time_val, peak_group in all_fits.groupby('peak_time'):
        matched, d_km, bearing = match_candidate(peak_group, lat, lon, la_0, lo_0)
        if len(matched) == 0:
            continue
        records.append({
            'candidate': name, 'peak_time': peak_time_val,
            'emission_median': matched['emission_mol'].median(),
            'emission_min': matched['emission_mol'].min(),
            'emission_max': matched['emission_mol'].max(),
            'emission_duration_min': matched['emission_duration_min'].mean(),
            'dist_km': d_km, 'bearing_deg': bearing, 'n_segments': len(matched),
            'wind_speed': matched[matched['source_agl_segmentaion']==met_agl]['wind_speed'].mean(),
            'wind_speed_std': matched[matched['source_agl_segmentaion']==met_agl]['wind_speed'].std(),
            'wind_dir_deg': matched[matched['source_agl_segmentaion']==met_agl]['wind_dir_deg'].mean(), # CAUTION no cyclic mean !
            'wind_dir_deg_std': matched[matched['source_agl_segmentaion']==met_agl]['wind_dir_deg'].std(), # CAUTION no cyclic mean!
            
        })

fig5_data = pd.DataFrame.from_records(records)
print(f"{len(fig5_data)} (candidate, peak) matches across {fig5_data['candidate'].nunique()} candidates "
      f"and {fig5_data['peak_time'].nunique()} of {all_fits['peak_time'].nunique()} processed peaks."
      if len(fig5_data) else "No matches found.")
fig5_data.sort_values(['peak_time', 'candidate'])

91 (candidate, peak) matches across 7 candidates and 13 of 13 processed peaks.


,candidate,peak_time,emission_median,emission_min,emission_max,emission_duration_min,dist_km,bearing_deg,n_segments,wind_speed,wind_speed_std,wind_dir_deg,wind_dir_deg_std
78,LBNL1,986.300000,1286.111503,1103.494766,12067.886214,0.330416,0.786946,79.254628,8,6.676654,NaN,58.459389,NaN
26,NG infrastructure,986.300000,374.093213,334.211512,729.476228,0.577341,0.263551,95.269514,9,6.289849,NaN,58.961703,NaN
39,building 30,986.300000,1742.439563,1482.901257,9914.052041,0.304825,0.910776,84.985044,7,NaN,NaN,NaN,NaN
52,building 33,986.300000,1742.439563,1482.901257,9914.052041,0.304825,0.937936,87.867733,7,NaN,NaN,NaN,NaN
0,foothill dining,986.300000,105.564223,69.850589,881.646135,0.621121,0.116357,108.362127,10,6.030064,NaN,59.891496,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,building 30,1240.266667,10364.283465,2387.085950,27209.899444,0.514359,0.910776,84.985044,24,5.748019,0.086636,57.451167,11.677763
64,building 33,1240.266667,10364.283465,2387.085950,27209.899444,0.514359,0.937936,87.867733,24,5.748019,0.086636,57.451167,11.677763
12,foothill dining,1240.266667,163.690706,138.884868,452.769762,0.769422,0.116357,108.362127,10,4.978723,NaN,57.828589,NaN
25,student housing,1240.266667,210.422905,77.470974,485.456688,0.775997,0.091360,68.320177,10,4.952048,NaN,57.737926,NaN


In [15]:
M_CH4 = 16.04  # g/mol, molare Masse Methan
SECONDS_PER_YEAR = 365.25 * 24 * 3600


def convert_emissions(
    df,
    mol_cols=('emission_median', 'emission_min', 'emission_max'),
    duration_col='emission_duration_min',          # z.B. 'duration_minutes', falls pro Zeile vorhanden
    duration_minutes=None,      # alternativ: feste Dauer für alle Zeilen (z.B. CONFIG['peak_window_minutes'])
    m_ch4=M_CH4,
    inplace=False,
):
    """
    Rechnet CH4-Emissionen von mol/Peak in g/s, kg/Peak und tCH4/yr um.

    Parameters
    ----------
    df : DataFrame mit Spalten in `mol_cols` (Einheit: mol pro Peak-Ereignis)
    duration_col : Spaltenname mit Peak-Dauer in Minuten (row-wise), optional
    duration_minutes : feste Peak-Dauer in Minuten, falls kein duration_col vorhanden
    m_ch4 : molare Masse CH4 in g/mol (default 16.04)
    inplace : bei True wird df direkt erweitert, sonst Kopie zurückgegeben

    Returns
    -------
    DataFrame mit zusätzlichen Spalten <col>_g_s, <col>_kg_peak, <col>_tCH4_yr
    """
    if duration_col is None and duration_minutes is None:
        raise ValueError("Entweder duration_col oder duration_minutes angeben (Peak-Dauer nötig für g/s).")

    out = df if inplace else df.copy()

    if duration_col is not None:
        dur_s = out[duration_col].astype(float) * 60.0
    else:
        dur_s = pd.Series(duration_minutes * 60.0, index=out.index)

    for col in mol_cols:
        mass_g = out[col] * m_ch4          # mol -> g (Gesamtmasse pro Peak)
        out[f'{col}_kg_peak'] = mass_g / 1000.0
        out[f'{col}_g_s'] = mass_g / dur_s
        out[f'{col}_tCH4_yr'] = out[f'{col}_g_s'] * SECONDS_PER_YEAR / 1e6  # g/s -> t/yr

    return out


# --- Beispielaufruf ---
# Fall A: feste Fensterbreite (z.B. gleich für alle Peaks, aus CONFIG)
result = convert_emissions(
    pd.DataFrame(records)  # oder dein oben gezeigter DataFrame
)


result_corr = apply_wind_speed_correction(
    result,
    met_path='.\demo_data_allpeaks\LBNL1_20161103.csv',
    reference_candidate='LBNL1',
)


result_corr[['candidate', 'peak_time', 'emission_median_g_s',
             'emission_median_g_s_wcorr',
             'wind_speed_corr_factor']].head()

unit_cols = [c for c in result_corr.columns if c.endswith(('_g_s_wcorr', '_kg_peak_wcorr', '_tCH4_yr_wcorr'))]
summary = result_corr.groupby('candidate')[unit_cols].agg(['mean', 'min', 'max'])
summary

emission_median_kg_peak_wcorr                         \
                                           mean        min         max   
candidate                                                                
NG infrastructure                     20.397518   2.147859  186.309991   
building 30                           95.819004  10.300568  441.292564   
building 33                           95.819004  10.300568  441.292564   
foothill dining                        3.682403   0.743933   27.316265   
student housing                        1.578037   0.627717    4.118542   
tank                                 170.916938  25.480552  478.105998   

                  emission_median_g_s_wcorr                             \
                                       mean          min           max   
candidate                                                                
NG infrastructure                358.715813    40.922150   2209.977321   
building 30                     2935.001283   317.449574   5501.203640   
building 33                     2935.001283   317.449574   5501.203640   
foothill dining                   68.892463    19.962100    444.823130   
student housing                   31.736983    10.602667     61.406723   
tank                            5638.761315  1379.691293  11778.887221   

                  emission_median_tCH4_yr_wcorr                               \
                                           mean           min            max   
candidate                                                                      
NG infrastructure                  11320.210152   1291.404834   69741.580304   
building 30                        92621.596497  10017.946667  173604.783985   
building 33                        92621.596497  10017.946667  173604.783985   
foothill dining                     2174.080803    629.955976   14037.550422   
student housing                     1001.543016    334.594730    1937.848806   
tank                              177945.774065  43539.745961  371713.411363   

                  emission_min_kg_peak_wcorr  ... emission_min_tCH4_yr_wcorr  \
                                        mean  ...                        max   
candidate                                     ...                              
NG infrastructure                   5.394249  ...               10647.003519   
building 30                        19.456799  ...               48837.591002   
building 33                        19.456799  ...               48837.591002   
foothill dining                     1.179669  ...                1509.619496   
student housing                     0.929881  ...                1370.672200   
tank                               31.814488  ...              113249.856277   

                  emission_max_kg_peak_wcorr                           \
                                        mean         min          max   
candidate                                                               
NG infrastructure                  71.380059    3.933553   701.825887   
building 30                       518.087215   69.866337  4286.488470   
building 33                       518.087215   69.866337  4286.488470   
foothill dining                    18.866818    2.921412   162.740257   
student housing                    25.332158    2.036821   277.803776   
tank                              947.263026  171.791730  7394.242319   

                  emission_max_g_s_wcorr                             \
                                    mean          min           max   
candidate                                                             
NG infrastructure            1192.007948    90.260374   8324.938893   
building 30                 11961.870259  1908.897635  51773.279245   
building 33                 11961.870259  1908.897635  51773.279245   
foothill dining               342.171736    77.569317   2650.092538   
student housing               418.407748    45.014050   4309.210908   
tank            

In [16]:
def summary_to_latex(summary, cols=None, caption="CH4-Emissionen je Kandidat", label="tab:ch4_emissions"):
    """
    Erwartet ein `summary`-DataFrame mit MultiIndex-Spalten (Ergebnis von
    result.groupby('candidate')[unit_cols].agg(['mean','min','max'])).
    """
    df = summary.copy()

    if cols is None:
        cols = [c for c in df.columns.get_level_values(0).unique()]

    # hübsche Spaltenüberschriften
    label_map = {
        'emission_median_g_s': 'g/s',
        'emission_median_kg_peak': 'kg/peak',
        'emission_median_tCH4_yr': 'tCH4/yr',
        'emission_min_g_s': 'g/s (min)',
        'emission_min_kg_peak': 'kg/peak (min)',
        'emission_min_tCH4_yr': 'tCH4/yr (min)',
        'emission_max_g_s': 'g/s (max)',
        'emission_max_kg_peak': 'kg/peak (max)',
        'emission_max_tCH4_yr': 'tCH4/yr (max)',
    }

    flat = pd.DataFrame(index=df.index)
    for col in cols:
        for stat in ['mean', 'min', 'max']:
            new_name = f"{label_map.get(col, col)} ({stat})"
            flat[new_name] = df[(col, stat)]

    flat = flat.reset_index()

    latex_table = flat.to_latex(
        index=False,
        float_format="%.2f",
        caption=caption,
        label=label,
        column_format='l' + 'r' * (len(flat.columns) - 1),
        escape=True,
    )
    return latex_table


print(summary_to_latex(summary, cols=['emission_median_g_s_wcorr', 'emission_median_kg_peak_wcorr', 'emission_median_tCH4_yr_wcorr']))

\begin{table}
\caption{CH4-Emissionen je Kandidat}
\label{tab:ch4_emissions}
\begin{tabular}{lrrrrrrrrr}
\toprule
candidate & emission\_median\_g\_s\_wcorr (mean) & emission\_median\_g\_s\_wcorr (min) & emission\_median\_g\_s\_wcorr (max) & emission\_median\_kg\_peak\_wcorr (mean) & emission\_median\_kg\_peak\_wcorr (min) & emission\_median\_kg\_peak\_wcorr (max) & emission\_median\_tCH4\_yr\_wcorr (mean) & emission\_median\_tCH4\_yr\_wcorr (min) & emission\_median\_tCH4\_yr\_wcorr (max) \\
\midrule
NG infrastructure & 358.72 & 40.92 & 2209.98 & 20.40 & 2.15 & 186.31 & 11320.21 & 1291.40 & 69741.58 \\
building 30 & 2935.00 & 317.45 & 5501.20 & 95.82 & 10.30 & 441.29 & 92621.60 & 10017.95 & 173604.78 \\
building 33 & 2935.00 & 317.45 & 5501.20 & 95.82 & 10.30 & 441.29 & 92621.60 & 10017.95 & 173604.78 \\
foothill dining & 68.89 & 19.96 & 444.82 & 3.68 & 0.74 & 27.32 & 2174.08 & 629.96 & 14037.55 \\
student housing & 31.74 & 10.60 & 61.41 & 1.58 & 0.63 & 4.12 & 1001.54 & 334.59 & 1937.85

In [17]:
UTC_OFFSET_HOURS = CONFIG['utc_offset_hours']
base = pd.to_datetime(CONFIG['date'])

fig5_data = result_corr.copy()

fig, ax = plt.subplots(figsize=(11, 5))

peak_times = sorted(fig5_data['peak_time'].unique())
candidates = list(SOURCE_LOCATIONS.keys())
n_cand = len(candidates)
bar_width = 0.8 / n_cand
colors = plt.cm.tab10(np.linspace(0, 1, n_cand))

for i, cand in enumerate(candidates):
    if cand in ['LBNL1','tank']:
        continue
    
    sub = fig5_data[fig5_data['candidate'] == cand].set_index('peak_time').reindex(peak_times)
    x = np.arange(len(peak_times)) + (i - n_cand / 2 + 0.5) * bar_width
    yerr = np.vstack([
        (sub['emission_median_wcorr'] - sub['emission_min_wcorr']).clip(lower=0).fillna(0),
        (sub['emission_max_wcorr'] - sub['emission_median_wcorr']).clip(lower=0).fillna(0),
    ])
    if len(sub.dist_km.dropna())==0:
        continue
    ax.bar(x, sub['emission_median_wcorr'].fillna(0), width=bar_width, color=colors[i], label=cand+' (%1.0fm)'%(sub.dist_km.dropna().iloc[0]*1000))
    ax.errorbar(x, sub['emission_median_wcorr'], yerr=yerr, fmt='none', ecolor='k', elinewidth=0.7, capsize=2)


def dual_time_formatter(x, pos):
    dt_utc = mdates.num2date(x).replace(tzinfo=None)
    dt_local = dt_utc + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    return f"{dt_utc.strftime('%H:%M')}\n({dt_local.strftime('%H:%M')} PDT)"

ax.set_yscale('log', base=10)
ax.set_ylim((10,4e5))
ax.set_xticks(np.arange(len(peak_times)))
xticklabels = [(base + pd.Timedelta(minutes=pt)).strftime('%H:%M') + (base + pd.Timedelta(minutes=pt+UTC_OFFSET_HOURS*60)).strftime('\n(%H:%M)') for pt in peak_times]
ax.set_xticklabels(xticklabels, rotation=90, ha='right')

ax.set_xlabel('peak time UTC (local)')
ax.set_ylabel('CH4 peak emission / mol')
ax.legend(fontsize=8, ncol=2)
ax.set_title('peak emissions per candidate')

ax2 = ax.twinx()
ax2.set_yscale('log', base=10)
ax2.set_ylim(np.array(ax.get_ylim()) * 16.04 / 1000)
ax2.set_ylabel('CH4 peak emission / kg')

plt.tight_layout()
plt.savefig('Figure5.png', dpi=250)
print('saved Figure5.png')

saved Figure5.png


In [18]:
import matplotlib.lines as mlines

HIST_POINTS_MIN = 500
peak_groups = sorted(all_fits['peaktime_grid'].unique())
n_peaks = len(peak_groups)
ncols, nrows = 2, 7

fig, axes = plt.subplots(nrows, ncols, figsize=(8.27, 11.69), sharex=True, sharey=True)  # A4 portrait
axes_flat = axes.flatten()

cols = ['recep_dist_km', 'residual_std_ppm', 'duration_residual_std_ppm']

for i, ptg in enumerate(peak_groups):
    ax = axes_flat[i]
    sub = fitted_by_peak[ptg]                       # CHANGED: full data incl. array cols
    peak_time = sub.peak_time.iloc[0]
    well = sub[sub['hist_points'] > HIST_POINTS_MIN].copy()
    ctime = base + pd.Timedelta(minutes=float(peak_time))

    if len(well) == 0:
        ax.text(0.5, 0.5, 'no well-sampled\nsegments', ha='center', va='center', fontsize=7, transform=ax.transAxes)
        ax.set_title(f"{ctime:%H:%M} UTC", fontsize=8)
        continue

    me = well.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanmean)
    st = well.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanstd)
    r_max = me.loc[me['residual_std_ppm'].idxmin(), 'recep_dist_km']

    well['dist_to_rmax'] = (well['recep_dist_km'] - r_max).abs()
    example_row = well.sort_values('dist_to_rmax').iloc[0]

    ax.errorbar(me['recep_dist_km'], me['duration_residual_std_ppm'] * 1000, yerr=st['duration_residual_std_ppm'] * 1000,
                fmt='-o', c='red', ms=2.5, lw=0.5, capsize=1.5, elinewidth=0.5)
    ax.errorbar(me['recep_dist_km'], me['residual_std_ppm'] * 1000, yerr=st['residual_std_ppm'] * 1000,
                fmt=':d', c='orange', ms=2.5, lw=0.5, capsize=1.5, elinewidth=0.5)
    ax.axvline(r_max, color='r', lw=0.6, ls='--')
    ax.set_title(f"{ctime:%H:%M} UTC, r$_{{max}}$={r_max:.2f} km", fontsize=8)
    ax.tick_params(labelsize=7)

    # kleines Inset-Fenster analog zu plot_figure4 (a01)
    ax_inset = ax.inset_axes([0.5, 0, 0.5, 0.5])
    t_ = example_row['time_minutes']
    ax_inset.plot(t_, example_row['observations_ppm'] + example_row['duration_residual'], c='red', lw=0.6, zorder=9)
    ax_inset.plot(t_, example_row['kernel_fitted'], c='orange', lw=0.6, zorder=10)
    ax_inset.plot(t_, example_row['observations_ppm'], linestyle=':', color='black', lw=0.6, zorder=11)
    ax_inset.get_xaxis().set_visible(False)
    ax_inset.get_yaxis().set_visible(False)

for j in range(n_peaks, nrows * ncols):
    axes_flat[j].axis('off')

if n_peaks < nrows * ncols:
    leg_ax = axes_flat[n_peaks]
    handles = [
        mlines.Line2D([], [], color='orange', ls=':', marker='d', ms=4, lw=0.8, label='kernel only'),
        mlines.Line2D([], [], color='red', ls='-', marker='o', ms=4, lw=0.8, label='kernel * emissions'),
        mlines.Line2D([], [], color='r', ls='--', lw=0.8, label=r'$r_{max}$'),
        mlines.Line2D([], [], color='black', ls=':', lw=0.8, label='observation (inset)'),
    ]
    leg_ax.legend(handles=handles, loc='center', fontsize=9, frameon=False)

fig.supxlabel('upwind distance / km', fontsize=9)
fig.supylabel('fit residual std / ppb', fontsize=9)
plt.tight_layout(rect=[0.02, 0.02, 1, 1])
plt.savefig('Figure_multi_peak_validation.png', dpi=250)
print(f"saved Figure_multi_peak_validation.png ({n_peaks} peaks)")


saved Figure_multi_peak_validation.png (13 peaks)


In [20]:
mes_df

,source_agl_segmentaion,recep_dist_segmentation,recep_bearing_segmentation,time,index,lati,long,zagl,zsfc,foot_no_hnf,...,kernel_fitted,duration_minutes,duration_residual,duration_residual_std_ppm,emission_mol,emission_duration_min,recep_bearing_deg_mid,peak_time,peaktime_grid,color_norm
0,"[0.0, 10.0)","[0.0, 0.01)","[-7.563, 139.462)",-0.023226,0.023226,37.875743,-122.257353,4.570715,233.960430,0.006677,...,"[0.000968507441673505, 0.000968507441673505, 0...","[-7.612802901055902e-11, -7.577466510823502e-1...","[0.0013435074416734264, 0.00026350744167344647...",0.000394,34.831287,0.821412,65.9495,1240.266667,1245.0,0.079503
2,"[0.0, 10.0)","[0.01, 0.022)","[0.489, 114.523)",-0.098418,0.098418,37.875799,-122.257238,5.308471,233.961906,0.006677,...,"[0.0009656524419961659, 0.0009656524419961659,...","[6.529114259499058e-10, 2.352445484370878e-09,...","[0.0013406524419973564, 0.00026065244200059503...",0.000393,68.827027,0.806597,57.5060,1240.266667,1245.0,0.076725
7,"[0.0, 10.0)","[0.022, 0.0364)","[31.735, 83.266)",-0.151751,0.151751,37.875868,-122.257111,5.657479,233.956725,0.006677,...,"[0.0009718110572965552, 0.0009718110572965552,...","[-8.2419040659896e-11, -8.064991871850596e-11,...","[0.0013468110572963715, 0.00026681105729639326...",0.000399,53.264838,0.835236,57.5005,1240.266667,1245.0,0.095463
13,"[0.0, 10.0)","[0.0364, 0.0537)","[40.203, 76.474)",-0.234043,0.234043,37.875942,-122.256954,5.915514,233.955625,0.006677,...,"[0.0009475033730842908, 0.0009475033730842908,...","[-9.999998796256403e-11, -9.99999784771339e-11...","[0.001322503373084232, 0.00024250337308424978,...",0.000378,208.824290,0.709133,58.3385,1240.266667,1245.0,0.026510
20,"[0.0, 10.0)","[0.0537, 0.0744)","[44.388, 73.616)",-0.330404,0.330404,37.876026,-122.256765,6.113373,233.981866,0.006677,...,"[0.0009400829852419463, 0.0009400829852419463,...","[1.1352459175905071e-08, 1.8240068347448162e-0...","[0.0013150829852516088, 0.00023508298525851652...",0.000377,246.678515,0.681972,59.0020,1240.266667,1245.0,0.023003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1529,"[90.0, 100.0)","[5.674, 6.819)","[53.911, 69.044)",-21.043836,21.043836,37.901865,-122.195084,94.942409,202.480389,0.005045,...,"[0.0002385906374342373, 0.0002385906374342373,...","[5.9887715774699044e-05, 0.0001002761885529197...","[0.0007754071206200423, 0.0002568461003506475,...",0.000672,118864.641354,0.301099,61.4775,1240.266667,1245.0,0.979079
1530,"[90.0, 100.0)","[5.674, 6.819)","[69.044, 84.177)",-22.396928,22.396928,37.890837,-122.189311,94.887219,214.249440,0.004938,...,"[0.00043551359637315643, 0.0004355135963731564...","[0.8270152164708866, 6.0842307146881796, 0.567...","[0.0009715607937074674, 0.00011270536337892106...",0.000613,298785.539325,0.323873,76.6105,1240.266667,1245.0,0.789344
1535,"[90.0, 100.0)","[6.819, 8.192)","[38.954, 54.075)",-26.271096,26.271096,37.919265,-122.193640,94.799265,211.113745,0.005486,...,"[0.0008269848528575591, 0.0007192532450147011,...","[0.0008008068612852488, 0.0011888299252731504,...","[0.001294617112696789, 2.6119601247497114e-05,...",0.000586,459729.308940,0.323350,46.5145,1240.266667,1245.0,0.700337
1536,"[90.0, 100.0)","[6.819, 8.192)","[54.075, 69.197)",-25.766977,25.766977,37.907148,-122.182633,94.945473,223.963701,0.004885,...,"[0.00012125046710971275, 0.0001470331616442593...","[3.394939767787457e-07, 4.820884890796429, 5.9...","[0.0007461018982709524, 0.0002716914416712843,...",0.000656,181101.472376,0.254952,61.6360,1240.266667,1245.0,0.928393


In [28]:
for k in fitted_by_peak.keys():
    print(k)
    print(base + pd.Timedelta(minutes=float(k)))
    
j=2


990.0
2016-11-03 16:30:00
1005.0
2016-11-03 16:45:00
1020.0
2016-11-03 17:00:00
1035.0
2016-11-03 17:15:00
1050.0
2016-11-03 17:30:00
1065.0
2016-11-03 17:45:00
1080.0
2016-11-03 18:00:00
1095.0
2016-11-03 18:15:00
1125.0
2016-11-03 18:45:00
1140.0
2016-11-03 19:00:00
1200.0
2016-11-03 20:00:00
1215.0
2016-11-03 20:15:00
1245.0
2016-11-03 20:45:00


In [39]:
#KMZ file erstellen für Figure 6
# Hilfsfunktion zum Umwandeln von RGB in KML-Farbformat (aabbggrr)
def rgba_to_kml_color(rgba):
    r, g, b, a = [int(255 * x) for x in rgba]
    return f'{a:02x}{b:02x}{g:02x}{r:02x}'

#quantile_target = CONFIG['quantile']
quantile_target = 'residual_std_ppm'
#quantile_target = 'duration_residual_std_ppm'
rnorm = 2 #km
loff, laff = (-0.00043, -0.0002)
loff, laff = (0, 0)
import simplekml
kml = simplekml.Kml()
cmap = plt.colormaps['jet']

#mes_df = fitted_to_obs[fitted_to_obs['hist_points'] > HIST_POINTS_MIN].copy()   # CHANGED: filtered -> fitted_to_obs, mit hist_points-Filter
for j in range(len(fitted_by_peak.keys())):
    mes_df = fitted_by_peak[list(fitted_by_peak.keys())[j]].copy()
    mes_df = mes_df[mes_df['hist_points']>HIST_POINTS_MIN]
    
    ctime = base + pd.Timedelta(minutes=float(mes_df['peak_time'].iloc[0]))

    mn = np.nanmin(mes_df[mes_df.recep_dist_km < rnorm][quantile_target])
    mx = np.nanmax(mes_df[mes_df.recep_dist_km < rnorm][quantile_target])#.quantile(0.95)
    mes_df['color_norm'] = ((mes_df[quantile_target] - mn) / (mx - mn))
    
    
    if False:
        plt.figure()
        plt.scatter(mes_df.recep_dist_km, mes_df[quantile_target])
        plt.axhline(mx,c='r')
        plt.axhline(mn,c='b')
        plt.xlabel('recep dist / km')
        plt.ylabel(quantile_target)

        plt.figure()
        plt.scatter(mes_df.recep_dist_km, mes_df['alt'],label='alt')
        plt.scatter(mes_df.recep_dist_km, mes_df['zagl'],label='zagl')
        plt.xlabel('recep dist / km')
        plt.ylabel('altitude / m')


    mes_df.loc[(mes_df.color_norm > 1), 'color_norm'] = 1


    rgba = cmap(0.99)  # RGBA-Wert aus Colormap
    kml_color = rgba_to_kml_color(rgba)  # Umwandlung zu KML-Format

    seg = kml.newpoint(coords=[(lo_0 + loff, la_0 + laff, 2)])
    seg.altitudemode = simplekml.AltitudeMode.relativetoground
    seg.style.iconstyle.scale = 3  # Icon thrice as big
    seg.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/shapes/dot.png'
    seg.style.iconstyle.color = simplekml.Color.blueviolet


    for i in range(len(mes_df)):

        coords = [(mes_df.iloc[i].long + loff, mes_df.iloc[i].lati + laff, mes_df.iloc[i].zagl)]  # (Lon, Lat, Höhe)

        rgba = cmap(mes_df.color_norm.iloc[i])  # RGBA-Wert aus Colormap
        kml_color = rgba_to_kml_color(rgba)  # Umwandlung zu KML-Format

        seg = kml.newpoint(coords=coords)
        seg.altitudemode = simplekml.AltitudeMode.relativetoground
        seg.style.iconstyle.scale = 3  # Icon thrice as big
        seg.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/shapes/dot.png'
        seg.style.iconstyle.color = kml_color
        # seg.style.linestyle.width = 2

    del mes_df
    # Speichern als KMZ
    kmz_path = "./Source_fit_residual_%s.kmz"%(f"{ctime:%H:%M}_UTC".replace(':','_'))
    kml.savekmz(kmz_path)
    print('saved at: ', end='')
    print(kmz_path)

saved at: ./Source_fit_residual_16_26_UTC.kmz
saved at: ./Source_fit_residual_16_38_UTC.kmz
saved at: ./Source_fit_residual_17_02_UTC.kmz
saved at: ./Source_fit_residual_17_14_UTC.kmz
saved at: ./Source_fit_residual_17_26_UTC.kmz
saved at: ./Source_fit_residual_17_37_UTC.kmz
saved at: ./Source_fit_residual_18_02_UTC.kmz
saved at: ./Source_fit_residual_18_13_UTC.kmz
saved at: ./Source_fit_residual_18_48_UTC.kmz
saved at: ./Source_fit_residual_19_00_UTC.kmz
saved at: ./Source_fit_residual_19_56_UTC.kmz
saved at: ./Source_fit_residual_20_15_UTC.kmz
saved at: ./Source_fit_residual_20_40_UTC.kmz


In [30]:
for k in mes_df.keys():
    print(k)

source_agl_segmentaion
recep_dist_segmentation
recep_bearing_segmentation
time
index
lati
long
zagl
zsfc
foot_no_hnf
samt
sigw
tlgr
mlht
dens
zasl
sigma
plume_height
foot
inmh
inmh_above_pbl
distance_m
dt_sec
wind_speed
wind_dir_deg
indx
alt
scf
numpar
recep
ak
full_weight_per_trajectory
recep_dist_km
recep_bearing_deg
recep_dist_north_km
recep_dist_east_km
dz_source
dalpha
deg_step
segment_area_m2
foot_source
foot_recep
group_vertical_weight
time_std
index_std
lati_std
long_std
zagl_std
zsfc_std
foot_no_hnf_std
samt_std
sigw_std
tlgr_std
mlht_std
dens_std
zasl_std
sigma_std
plume_height_std
foot_std
inmh_std
inmh_above_pbl_std
distance_m_std
dt_sec_std
wind_speed_std
wind_dir_deg_std
indx_std
alt_std
scf_std
numpar_std
recep_std
ak_std
full_weight_per_trajectory_std
recep_dist_km_std
recep_bearing_deg_std
recep_dist_north_km_std
recep_dist_east_km_std
dz_source_std
dalpha_std
deg_step_std
segment_area_m2_std
foot_source_std
foot_recep_std
group_vertical_weight_std
residual_std_ppm
res

In [38]:
j=2
mes_df = fitted_by_peak[list(fitted_by_peak.keys())[j]].copy()
mes_df = mes_df[mes_df['hist_points']>HIST_POINTS_MIN]
mes_df = mes_df[(mes_df['recep_dist_east_km']<0.2) & (mes_df[quantile_target]>0.00035)]

plt.figure()
plt.scatter(mes_df.recep_dist_km,mes_df[quantile_target])

plt.figure()
for i in range(10):
    plt.plot(mes_df.iloc[i]['time_minutes'],mes_df.iloc[i]['duration_minutes'])

In [46]:
# KMZ file erstellen für Figure 6
# ------------------------------------------------------------
# Änderungen gegenüber der Vorversion (RC1-Kommentar zu Fig. 6):
#   1) Colormap 'jet' -> 'cool': durchgehend helle, leuchtende Farben
#      (Cyan -> Magenta). 'jet' beginnt bei color_norm=0 (= beste Fits,
#      die wichtigsten Punkte!) mit fast schwarzem Dunkelblau, das auf
#      Satellitenbild/Terrain in Google Earth kaum zu erkennen ist.
#      'cool' hat an keiner Stelle einen dunklen Bereich.
#   2) Site-/Receptor-Marker 'blueviolet' -> 'yellow': heller, höherer
#      Kontrast zu Terrain UND zur Punktwolken-Farbskala.
#   3) Farbskala wird zusätzlich als eigenes ScreenOverlay-Bild ins KMZ
#      eingebettet (Legende, fix in einer Bildschirmecke, unabhängig vom
#      3D-Blickwinkel) -> siehe erzeuge_colorbar_png() weiter unten.
#   4) Neu: SOURCE_LOCATIONS (die 5 Kandidaten + tank/LBNL1) werden als
#      eigener, klar erkennbarer Marker-Satz mit Namens-Label ergänzt,
#      getrennt vom Fit-Punkt-Datensatz.
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import simplekml
import os

# ------------------------------------------------------------
# 0) Kandidaten-Locations
# ------------------------------------------------------------
# (lat, lon, Kurzbeschreibung). 'tank' und 'LBNL1' werden im Manuskript
# nicht als Kandidaten geführt (tank: keine Substanz in dieser Revision,
# LBNL1: Standort der Meteorologie-Station, kein Emissions-Kandidat) und
# daher standardmäßig nicht als Kandidaten-Pins ausgegeben (siehe
# CANDIDATES_TO_PLOT unten). Beide bleiben im Dict verfügbar, falls sie
# später doch gebraucht werden (z.B. LBNL1 als Referenzpunkt).
SOURCE_LOCATIONS = {
    'foothill dining':   (37.87544915758012, -122.25616344191205, 'potential candidate, very close'),
    'student housing':   (37.87608233036625, -122.25645436619163, 'potential candidate, very close, broken heating system reported'),
    'NG infrastructure': (37.87556109517469, -122.25443161396028, 'natural gas valve, surrounded by trees, injection into atmosphere not likely'),
    'building 30':       (37.87649436515283, -122.24708452824339, 'smoke stack approximately 25m above ground'),
    'building 33':       (37.87609216845105, -122.24674281410584, 'smoke stack approximately 40m above ground'),
    'tank':              (37.87846958360055, -122.24143278375334, 'unknown tank, potential large source, injection into atmosphere possible due to height'),
    'LBNL1':             (37.877097985303465, -122.24861278713782, 'location of meteorological site, no source candidate'),
}

# Nur die fünf im Manuskript diskutierten Kandidaten als Pins ausgeben.
# Auf True setzen für tank/LBNL1, falls gewünscht.
CANDIDATES_TO_PLOT = [
    'foothill dining',
    'student housing',
    'NG infrastructure',
    'building 30',
    'building 33',
    'LBNL1'
]

# Helle, gut unterscheidbare Marker-Farbe für die Kandidaten (KML aabbggrr).
# Gelb sticht sowohl gegen das Terrain als auch gegen die cool-Farbskala
# (Cyan/Blau/Magenta) der Fit-Punktwolke klar hervor.
CANDIDATE_COLOR = simplekml.Color.yellow
RECEPTOR_COLOR = simplekml.Color.yellow  # einheitlich mit Kandidaten, ebenfalls hell statt blueviolet

CANDIDATE_ICON = 'http://maps.google.com/mapfiles/kml/paddle/ylw-stars.png'
FIT_POINT_ICON = 'http://maps.google.com/mapfiles/kml/shapes/dot.png'
RECEPTOR_ICON = 'http://maps.google.com/mapfiles/kml/shapes/star.png'


# ------------------------------------------------------------
# 1) Hilfsfunktionen
# ------------------------------------------------------------
def rgba_to_kml_color(rgba):
    """RGB(A) (0..1 floats) -> KML-Farbstring im Format aabbggrr."""
    r, g, b, a = [int(255 * x) for x in rgba]
    return f'{a:02x}{b:02x}{g:02x}{r:02x}'


def make_colorbar_png(path, cmap_name='cool', label='normalised fit residual\n(0 = best fit, 1 = worst)'):
    """Erzeugt eine eigenständige Farbskalen-Grafik (weißer Hintergrund,
    guter Kontrast) zum Einbetten als KML ScreenOverlay."""
    fig, ax = plt.subplots(figsize=(1.4, 4.2), dpi=200)
    cmap = plt.colormaps[cmap_name]
    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='vertical')
    cb.set_label(label, fontsize=9, color='black')
    cb.ax.tick_params(labelsize=8, color='black', labelcolor='black')
    cb.outline.set_edgecolor('black')
    cb.outline.set_linewidth(1.2)
    fig.patches.append(plt.Rectangle((0, 0), 1, 1, transform=fig.transFigure,
                                      facecolor='white', alpha=0.9, zorder=-1))
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight', pad_inches=0.15)
    plt.close(fig)
    return path


def add_colorbar_overlay(kml, png_path):
    """Bindet die Farbskala als ScreenOverlay ein (fixe Position oben
    rechts im Google-Earth-Fenster, unabhängig vom Blickwinkel)."""
    overlay = kml.newscreenoverlay(name='CH4 fit residual scale')
    overlay.icon.href = kml.addfile(png_path)
    # Position: oben rechts
    overlay.overlayxy = simplekml.OverlayXY(x=1, y=1, xunits=simplekml.Units.fraction, yunits=simplekml.Units.fraction)
    overlay.screenxy = simplekml.ScreenXY(x=0.98, y=0.98, xunits=simplekml.Units.fraction, yunits=simplekml.Units.fraction)
    overlay.size = simplekml.Size(x=0.14, y=0, xunits=simplekml.Units.fraction, yunits=simplekml.Units.fraction)
    return overlay


def add_candidate_markers(kml, locations, keys, color, icon_href):
    """Fügt die Source-Kandidaten als eigene, klar beschriftete Marker
    hinzu (Name dauerhaft sichtbar, Beschreibung im Infofenster)."""
    folder = kml.newfolder(name='Candidate source locations')
    for key in keys:
        lat, lon, desc = locations[key]
        distance = haversine_distance(lat,lon,la_0,lo_0)
        size_factor = max([1, distance/400])
        pnt = folder.newpoint(name=key, coords=[(lon, lat, 2)])
        pnt.description = desc
        pnt.altitudemode = simplekml.AltitudeMode.relativetoground
        pnt.style.iconstyle.scale = 1.8*size_factor
        pnt.style.iconstyle.icon.href = icon_href
        pnt.style.iconstyle.color = color
        pnt.style.labelstyle.scale = 1.5*size_factor
        pnt.style.labelstyle.color = simplekml.Color.white
    return folder


# ------------------------------------------------------------
# 2) Hauptroutine (Struktur wie im Originalskript, minimal-invasiv angepasst)
# ------------------------------------------------------------
def build_kmz(fitted_by_peak, base, lo_0, la_0,
              quantile_target='duration_residual_std_ppm',
              rnorm=2, loff=0, laff=0,
              hist_points_min=None, out_dir='.'):
    """
    fitted_by_peak : dict[str, DataFrame]  (wie im Originalskript)
    base            : pd.Timestamp (Referenzzeit für ctime)
    lo_0, la_0      : Receptor-Koordinaten (Lon, Lat)
    hist_points_min : entspricht HIST_POINTS_MIN im Originalskript
    """
    os.makedirs(out_dir, exist_ok=True)
    colorbar_png = make_colorbar_png(os.path.join(out_dir, 'colorbar_legend.png'))

    cmap = plt.colormaps['cool']  # <- Änderung 1: jet -> cool

    saved_paths = []
    for j in range(len(fitted_by_peak.keys())):
        key = list(fitted_by_peak.keys())[j]
        mes_df = fitted_by_peak[key].copy()
        if hist_points_min is not None:
            mes_df = mes_df[mes_df['hist_points'] > hist_points_min]

        ctime = base + pd.Timedelta(minutes=float(mes_df['peak_time'].iloc[0]))

        mn = np.nanmin(mes_df[mes_df.recep_dist_km < rnorm][quantile_target])
        mx = np.nanmax(mes_df[mes_df.recep_dist_km < rnorm][quantile_target])
        mes_df['color_norm'] = (mes_df[quantile_target] - mn) / (mx - mn)
        mes_df.loc[(mes_df.color_norm > 1), 'color_norm'] = 1

        kml = simplekml.Kml()

        # --- Farbskalen-Legende (Änderung 3) ---
        add_colorbar_overlay(kml, colorbar_png)

        # --- Receptor-Marker (Änderung 2: hellere Farbe) ---
        seg = kml.newpoint(name='Receptor', coords=[(lo_0 + loff, la_0 + laff, 2)])
        seg.altitudemode = simplekml.AltitudeMode.relativetoground
        seg.style.iconstyle.scale = 3
        seg.style.iconstyle.icon.href = RECEPTOR_ICON
        seg.style.iconstyle.color = RECEPTOR_COLOR

        # --- Fit-Punktwolke (Änderung 1: cool statt jet) ---
        fit_folder = kml.newfolder(name='Fit residual points')
        for i in range(len(mes_df)):
            coords = [(mes_df.iloc[i].long + loff, mes_df.iloc[i].lati + laff, mes_df.iloc[i].zagl)]
            rgba = cmap(mes_df.color_norm.iloc[i])
            kml_color = rgba_to_kml_color(rgba)

            seg = fit_folder.newpoint(coords=coords)
            seg.altitudemode = simplekml.AltitudeMode.relativetoground
            seg.style.iconstyle.scale = 3
            seg.style.iconstyle.icon.href = FIT_POINT_ICON
            seg.style.iconstyle.color = kml_color

        # --- Candidate-Marker (Änderung 4: neu) ---
        add_candidate_markers(kml, SOURCE_LOCATIONS, CANDIDATES_TO_PLOT,
                               CANDIDATE_COLOR, CANDIDATE_ICON)

        del mes_df
        kmz_path = os.path.join(out_dir, "Source_fit_residual_%s.kmz" % (f"{ctime:%H:%M}_UTC".replace(':', '_')))
        kml.savekmz(kmz_path)
        saved_paths.append(kmz_path)
        print('saved at: ', end='')
        print(kmz_path)

    return saved_paths

build_kmz(fitted_by_peak, base, lo_0, la_0,hist_points_min=HIST_POINTS_MIN, out_dir='./kmz_out')

if __name__ == '__main__':
    print("Dies ist ein Modul mit build_kmz(); bitte mit den bestehenden")
    print("fitted_by_peak / base / lo_0 / la_0 Variablen aus der Analyse aufrufen, z.B.:")
    print("")
    print("    from make_figure6_kmz import build_kmz, SOURCE_LOCATIONS")
    print("    build_kmz(fitted_by_peak, base, lo_0, la_0,")
    print("              hist_points_min=HIST_POINTS_MIN, out_dir='./kmz_out')")

saved at: ./kmz_out\Source_fit_residual_16_26_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_16_38_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_17_02_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_17_14_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_17_26_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_17_37_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_18_02_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_18_13_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_18_48_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_19_00_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_19_56_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_20_15_UTC.kmz
saved at: ./kmz_out\Source_fit_residual_20_40_UTC.kmz
Dies ist ein Modul mit build_kmz(); bitte mit den bestehenden
fitted_by_peak / base / lo_0 / la_0 Variablen aus der Analyse aufrufen, z.B.:

    from make_figure6_kmz import build_kmz, SOURCE_LOCATIONS
    build_kmz(fitted_by_peak, base, lo_0, la_0,
              hist_points_min=HIST_POINTS_MIN, o